In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from keras.datasets import fashion_mnist

In [2]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNet, InputLayer, DenseLayer, Sigmoid, Softmax, OneHotEncoder

In [3]:
print("Loading Fashion MNIST data...")
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=2
)
print("Done!")

Loading Fashion MNIST data...
Done!


In [4]:
# Flatten images and transpose: each column represents one flattened image
train_features = train_images.reshape(train_images.shape[0], -1).T
val_features = val_images.reshape(val_images.shape[0], -1).T
test_features = test_images.reshape(test_images.shape[0], -1).T

In [5]:
# Normalize pixel values to [0, 1]
train_features = train_features / 255.0
val_features = val_features / 255.0
test_features = test_features / 255.0

In [6]:
train_features = train_features[:, :2000]
val_features = val_features[:, :500]
test_features = test_features[:, :250]

In [7]:
train_labels_subset = train_labels[:2000]
val_labels_subset = val_labels[:500]
test_labels_subset = test_labels[:250]

In [8]:
encoder = OneHotEncoder()
encoder.fit(train_labels_subset, 10)
train_targets = encoder.transform(train_labels_subset)
val_targets = encoder.transform(val_labels_subset)
test_targets = encoder.transform(test_labels_subset)

In [9]:
network_layers = [
    InputLayer(data=train_features),
    DenseLayer(units=64, activation=Sigmoid(), name="HiddenLayer1"),
    DenseLayer(units=10, activation=Softmax(), name="OutputLayer")
]

In [10]:
model = NeuralNet(
    layers=network_layers,
    batch_size=2000,
    optimizer_name="SGD",
    init_method="Random",
    epochs=100,
    targets=train_targets,
    loss_type="CrossEntropy",
    X_val=val_features,
    targets_val=val_targets,
    use_wandb=False
)

In [11]:
model.forward_pass()
initial_output = model.layers[-1].output

In [12]:
history = model.backward_pass()

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 34.17it/s]


In [13]:
val_acc, val_loss, _ = model.evaluate(val_features, val_targets)
test_acc, test_loss, _ = model.evaluate(test_features, test_targets)

In [14]:
print("Training Results:")
# Calculate initial and final accuracy on the training
initial_accuracy = np.mean(np.argmax(initial_output, axis=0) == train_labels_subset)
final_accuracy = np.mean(np.argmax(model.layers[-1].output, axis=0) == train_labels_subset)
print("Untrained network accuracy on training ", initial_accuracy)
print("Trained network accuracy on training ", final_accuracy)

print("Validation Results:")
print("Validation accuracy:", val_acc / val_targets.shape[1])
print("Validation loss:", val_loss)

print("Test Results:")
print("Test accuracy:", test_acc / test_targets.shape[1])
print("Test loss:", test_loss)

Training Results:
Untrained network accuracy on training  0.0395
Trained network accuracy on training  0.382
Validation Results:
Validation accuracy: 0.348
Validation loss: 1056.286604960501
Test Results:
Test accuracy: 0.352
Test loss: 529.6112635110007
